In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
train_df = pd.read_csv("../splits/train.csv")
val_df = pd.read_csv("../splits/val.csv")

print("Train images:", len(train_df))
print("Validation images:", len(val_df))

Train images: 8017
Validation images: 1998


In [3]:
train_df = train_df.drop(columns=["image_path"], errors="ignore")
val_df = val_df.drop(columns=["image_path"], errors="ignore")

print(train_df.columns)

Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization'], dtype='str')


In [4]:
shared_lesions = set(train_df["lesion_id"]) & set(val_df["lesion_id"])

print("Shared lesions:", len(shared_lesions))

print("Total images:", len(train_df) + len(val_df))
print("Train percentage:", len(train_df) / (len(train_df) + len(val_df)) * 100)
print("Validation percentage:", len(val_df) / (len(train_df) + len(val_df)) * 100)

Shared lesions: 0
Total images: 10015
Train percentage: 80.0499251123315
Validation percentage: 19.9500748876685


In [5]:
IMAGE_DIR = Path("../HAM10000_images")

missing_train = []

for image_id in train_df["image_id"]:
    image_path = IMAGE_DIR / f"{image_id}.jpg"

    if not image_path.exists():
        missing_train.append(image_id)


missing_val = []

for image_id in val_df["image_id"]:
    image_path = IMAGE_DIR / f"{image_id}.jpg"

    if not image_path.exists():
        missing_val.append(image_id)


print("Missing train images:", len(missing_train))
print("Missing validation images:", len(missing_val))

Missing train images: 0
Missing validation images: 0


In [6]:
train_df.to_csv("../splits/train.csv", index=False)
val_df.to_csv("../splits/val.csv", index=False)

print("Corrected train.csv and val.csv saved!")

Corrected train.csv and val.csv saved!


phase 5: physically seperate the images


In [7]:
import shutil
from pathlib import Path

BASE_DIR = Path("..")

IMAGE_DIR = BASE_DIR / "HAM10000_images"
OUTPUT_DIR = BASE_DIR / "dataset"

classes = sorted(train_df["dx"].unique())

for split_name in ["train", "val"]:
    for class_name in classes:
        folder = OUTPUT_DIR / split_name / class_name
        folder.mkdir(parents=True, exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


copy training images


In [8]:
missing_train = []

for _, row in train_df.iterrows():

    image_name = row["image_id"] + ".jpg"
    class_name = row["dx"]

    source = IMAGE_DIR / image_name

    destination = (
        OUTPUT_DIR
        / "train"
        / class_name
        / image_name
    )

    if source.exists():
        shutil.copy2(source, destination)
    else:
        missing_train.append(image_name)

print("Missing training images:", len(missing_train))

Missing training images: 0


copy validation images


In [10]:
missing_val = []

for _, row in val_df.iterrows():

    image_name = row["image_id"] + ".jpg"
    class_name = row["dx"]

    source = IMAGE_DIR / image_name

    destination = (
        OUTPUT_DIR
        / "val"
        / class_name
        / image_name
    )

    if source.exists():
        shutil.copy2(source, destination)
    else:
        missing_val.append(image_name)

print("Missing validation images:", len(missing_val))

Missing validation images: 0


verify


In [13]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..")

DATASET_DIR = BASE_DIR / "dataset"

classes = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc"
]

for split in ["train", "val"]:

    print(f"\n{'='*40}")
    print(split.upper())
    print("="*40)

    total = 0

    for cls in classes:

        class_dir = DATASET_DIR / split / cls

        count = len(list(class_dir.glob("*.jpg")))

        total += count

        print(f"{cls:6s}: {count}")

    print("-" * 40)
    print("TOTAL:", total)


TRAIN
akiec : 257
bcc   : 412
bkl   : 891
df    : 86
mel   : 890
nv    : 5367
vasc  : 114
----------------------------------------
TOTAL: 8017

VAL
akiec : 70
bcc   : 102
bkl   : 208
df    : 29
mel   : 223
nv    : 1338
vasc  : 28
----------------------------------------
TOTAL: 1998


In [14]:
train_df = pd.read_csv("../splits/train.csv")
val_df = pd.read_csv("../splits/val.csv")

for split_name, dataframe in [
    ("train", train_df),
    ("val", val_df)
]:

    print(f"\n{'='*50}")
    print(split_name.upper())
    print("="*50)

    for cls in classes:

        expected = (dataframe["dx"] == cls).sum()

        actual = len(
            list(
                (DATASET_DIR / split_name / cls).glob("*.jpg")
            )
        )

        status = "PASS" if expected == actual else "FAIL"

        print(
            f"{cls:6s} | "
            f"CSV: {expected:4d} | "
            f"Folder: {actual:4d} | "
            f"{status}"
        )


TRAIN
akiec  | CSV:  257 | Folder:  257 | PASS
bcc    | CSV:  412 | Folder:  412 | PASS
bkl    | CSV:  891 | Folder:  891 | PASS
df     | CSV:   86 | Folder:   86 | PASS
mel    | CSV:  890 | Folder:  890 | PASS
nv     | CSV: 5367 | Folder: 5367 | PASS
vasc   | CSV:  114 | Folder:  114 | PASS

VAL
akiec  | CSV:   70 | Folder:   70 | PASS
bcc    | CSV:  102 | Folder:  102 | PASS
bkl    | CSV:  208 | Folder:  208 | PASS
df     | CSV:   29 | Folder:   29 | PASS
mel    | CSV:  223 | Folder:  223 | PASS
nv     | CSV: 1338 | Folder: 1338 | PASS
vasc   | CSV:   28 | Folder:   28 | PASS


In [15]:
shared_images = (
    set(train_df["image_id"])
    &
    set(val_df["image_id"])
)

print("Shared images:", len(shared_images))

assert len(shared_images) == 0

print("PASS: No duplicate images between train and validation.")

Shared images: 0
PASS: No duplicate images between train and validation.


In [16]:
shared_lesions = (
    set(train_df["lesion_id"])
    &
    set(val_df["lesion_id"])
)

print("Shared lesions:", len(shared_lesions))

assert len(shared_lesions) == 0

print("PASS: No lesion-level data leakage.")

Shared lesions: 0
PASS: No lesion-level data leakage.


In [17]:
from PIL import Image

def check_corrupted_images(folder):

    corrupted = []

    for image_path in folder.rglob("*.jpg"):

        try:
            with Image.open(image_path) as img:
                img.verify()

        except Exception:
            corrupted.append(str(image_path))

    return corrupted

In [18]:
corrupted_train = check_corrupted_images(
    DATASET_DIR / "train"
)

corrupted_val = check_corrupted_images(
    DATASET_DIR / "val"
)

print("Corrupted train images:", len(corrupted_train))
print("Corrupted validation images:", len(corrupted_val))

Corrupted train images: 0
Corrupted validation images: 0
